# Step 1: Data Cleaning (Python)

**Project:** Supply Chain Late-Delivery Diagnostic (DataCo)

**Goal of this notebook:** turn the raw file into clean tables that MySQL can analyse.

| | |
|---|---|
| **Input** | `data/raw/DataCoSupplyChainDataset.csv` (180,519 rows, 53 columns) |
| **Output** | `data/processed/orders_clean.csv` (one row per order) and `data/processed/order_lines_clean.csv` (one row per product line) |
| **Next step** | load both files into MySQL (`sql/00_create_tables_and_load.sql`) |

All business analysis is done later in SQL. This notebook only **checks** and **cleans**.

## 1. Load the raw data

In [1]:
import pandas as pd

# The file is saved in Latin-1 (not UTF-8), so we must say so, otherwise accented city names break
df = pd.read_csv("../data/raw/DataCoSupplyChainDataset.csv", encoding="latin-1")

print("Rows, columns:", df.shape)

Rows, columns: (180519, 53)


In [2]:
df.head(3)

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


## 2. Understand the data
One row = one **order line** (one product inside an order). An order with 3 products has 3 rows.

In [3]:
print("Order lines :", len(df))
print("Orders      :", df["Order Id"].nunique())
print("Customers   :", df["Customer Id"].nunique())
print("Products    :", df["Product Name"].nunique())

Order lines : 180519
Orders      : 65752
Customers   : 20652
Products    : 118


In [4]:
df["Delivery Status"].value_counts()

Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

In [5]:
df["Shipping Mode"].value_counts()

Shipping Mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737
Name: count, dtype: int64

## 3. Check for missing values

In [6]:
missing = df.isnull().sum()
missing[missing > 0]

Customer Lname              8
Customer Zipcode            3
Order Zipcode          155679
Product Description    180519
dtype: int64

Only 4 columns have missing values, and we don't need any of them
(`Order Zipcode`, `Product Description`, `Customer Lname`, `Customer Zipcode`). Nothing to fix.

## 4. Check for duplicate rows

In [7]:
print("Duplicate order lines:", df["Order Item Id"].duplicated().sum())

Duplicate order lines: 0


## 5. Keep only the columns we need
53 columns is too many. We keep 18 and drop the rest, including **personal data**
(customer names, street, email, password), which the analysis doesn't need.

In [8]:
keep = [
    "Order Id", "Order Item Id", "order date (DateOrders)",
    "Shipping Mode", "Days for shipment (scheduled)", "Days for shipping (real)",
    "Delivery Status", "Late_delivery_risk",
    "Market", "Order Region", "Order Country", "Customer Segment",
    "Department Name", "Category Name", "Product Name",
    "Order Item Quantity", "Order Item Total", "Order Profit Per Order",
]
df = df[keep]
print("Columns now:", df.shape[1])

Columns now: 18


## 6. Rename columns to simple names
Short names without spaces are much easier to use in SQL.

In [9]:
df = df.rename(columns={
    "Order Id": "order_id",
    "Order Item Id": "order_item_id",
    "order date (DateOrders)": "order_date",
    "Shipping Mode": "shipping_mode",
    "Days for shipment (scheduled)": "promised_days",
    "Days for shipping (real)": "actual_days",
    "Delivery Status": "delivery_status",
    "Late_delivery_risk": "is_late",
    "Market": "market",
    "Order Region": "order_region",
    "Order Country": "order_country",
    "Customer Segment": "customer_segment",
    "Department Name": "department_name",
    "Category Name": "category_name",
    "Product Name": "product_name",
    "Order Item Quantity": "quantity",
    "Order Item Total": "revenue",          # what the customer paid, after discount
    "Order Profit Per Order": "profit",     # profit on this line (the original name is misleading)
})
df.head(3)

,order_id,order_item_id,order_date,shipping_mode,promised_days,actual_days,delivery_status,is_late,market,order_region,order_country,customer_segment,department_name,category_name,product_name,quantity,revenue,profit
0,77202,180517,1/31/2018 22:56,Standard Class,4,3,Advance shipping,0,Pacific Asia,Southeast Asia,Indonesia,Consumer,Fitness,Sporting Goods,Smart watch,1,314.640015,91.250000
1,75939,179254,1/13/2018 12:27,Standard Class,4,5,Late delivery,1,Pacific Asia,South Asia,India,Consumer,Fitness,Sporting Goods,Smart watch,1,311.359985,-249.089996
2,75938,179253,1/13/2018 12:06,Standard Class,4,4,Shipping on time,0,Pacific Asia,South Asia,India,Consumer,Fitness,Sporting Goods,Smart watch,1,309.720001,-247.779999


## 7. Convert the date from text to a real date
The file stores dates as text like `1/31/2018 22:56`.

In [10]:
df["order_date"] = pd.to_datetime(df["order_date"], format="%m/%d/%Y %H:%M")

print("First order:", df["order_date"].min())
print("Last order :", df["order_date"].max())

First order: 2015-01-01 00:00:00
Last order : 2018-01-31 23:38:00


## 8. Check what "late" means
`is_late` should be 1 when the actual days are more than the promised days. Let's check every row.

In [11]:
my_late = (df["actual_days"] > df["promised_days"]).astype(int)
different = df[my_late != df["is_late"]]

print("Rows where the late flag disagrees:", len(different))
different["delivery_status"].value_counts()

Rows where the late flag disagrees: 4423


delivery_status
Shipping canceled    4423
Name: count, dtype: int64

All the disagreements are **cancelled** orders: they have shipping days but are never marked late,
because they were never delivered. For every shipped order, **late = actual days > promised days**.

## 9. Remove cancelled orders
A cancelled order was never delivered, so it can't be late or on time. Keeping it would make the late rate look better than it is.

In [12]:
before = len(df)
df = df[df["delivery_status"] != "Shipping canceled"]

print("Lines before:", before)
print("Lines after :", len(df))
print("Removed     :", before - len(df))

Lines before: 180519
Lines after : 172765
Removed     : 7754


## 10. Check that shipping info is the same for every line of an order
Next we combine lines into orders. That is only safe if all lines of an order have the same
shipping mode, status and actual days. Each check should print **1**.

In [13]:
print("Max shipping modes per order :", df.groupby("order_id")["shipping_mode"].nunique().max())
print("Max delivery status per order:", df.groupby("order_id")["delivery_status"].nunique().max())
print("Max actual days per order    :", df.groupby("order_id")["actual_days"].nunique().max())

Max shipping modes per order : 1
Max delivery status per order: 1
Max actual days per order    : 1


## 11. Table 1: order lines (for the product / department analysis)
One row per product line. Used in SQL to check if some departments are late more often.

In [14]:
order_lines = df[["order_item_id", "order_id", "department_name", "category_name",
                  "product_name", "quantity", "revenue", "profit", "is_late"]]

order_lines = order_lines.round({"revenue": 2, "profit": 2})
print("Order lines:", len(order_lines))
order_lines.head(3)

Order lines: 172765


,order_item_id,order_id,department_name,category_name,product_name,quantity,revenue,profit,is_late
0,180517,77202,Fitness,Sporting Goods,Smart watch,1,314.64,91.25,0
1,179254,75939,Fitness,Sporting Goods,Smart watch,1,311.36,-249.09,1
2,179253,75938,Fitness,Sporting Goods,Smart watch,1,309.72,-247.78,0


## 12. Table 2: orders (one row per order)
Delivery happens per **order**, so the main table has one row per order.
- shipping info is the same on every line, so we take the **first** value
- money and units are **added up** across the lines

In [15]:
orders = df.groupby("order_id").agg(
    order_date       = ("order_date", "first"),
    shipping_mode    = ("shipping_mode", "first"),
    promised_days    = ("promised_days", "first"),
    actual_days      = ("actual_days", "first"),
    delivery_status  = ("delivery_status", "first"),
    is_late          = ("is_late", "first"),
    market           = ("market", "first"),
    order_region     = ("order_region", "first"),
    order_country    = ("order_country", "first"),
    customer_segment = ("customer_segment", "first"),
    order_lines      = ("order_item_id", "count"),
    units            = ("quantity", "sum"),
    revenue          = ("revenue", "sum"),
    profit           = ("profit", "sum"),
).reset_index()

print("Orders:", len(orders))

Orders: 62897


## 13. Add one new column: days vs promise
`actual_days - promised_days`: positive = days late, negative = days early, 0 = exactly on time.

In [16]:
orders["days_vs_promise"] = orders["actual_days"] - orders["promised_days"]
orders = orders.round({"revenue": 2, "profit": 2})

# put the columns in the same order as the SQL table
orders = orders[["order_id", "order_date", "shipping_mode", "promised_days", "actual_days",
                 "days_vs_promise", "delivery_status", "is_late", "market", "order_region",
                 "order_country", "customer_segment", "order_lines", "units", "revenue", "profit"]]
orders.head()

,order_id,order_date,shipping_mode,promised_days,actual_days,days_vs_promise,delivery_status,is_late,market,order_region,order_country,customer_segment,order_lines,units,revenue,profit
0,1,2015-01-01 00:00:00,Standard Class,4,2,-2,Advance shipping,0,LATAM,Central America,México,Consumer,1,1,239.98,88.79
1,2,2015-01-01 00:21:00,Standard Class,4,3,-1,Advance shipping,0,LATAM,South America,Colombia,Consumer,3,7,529.38,195.90
2,4,2015-01-01 01:03:00,Standard Class,4,5,1,Late delivery,1,LATAM,South America,Colombia,Home Office,4,14,620.87,124.09
3,5,2015-01-01 01:24:00,Standard Class,4,6,2,Late delivery,1,LATAM,South America,Colombia,Consumer,5,10,987.07,390.09
4,7,2015-01-01 02:06:00,Second Class,2,3,1,Late delivery,1,LATAM,South America,Brasil,Consumer,3,7,525.52,203.93


## 14. Final checks

In [17]:
print("Orders            :", len(orders))
print("Missing values    :", orders.isnull().sum().sum())
print("Duplicate orders  :", orders["order_id"].duplicated().sum())
print("Late orders       :", orders["is_late"].sum())
print("Late rate         :", round(orders["is_late"].mean() * 100, 1), "%")

Orders            : 62897
Missing values    : 0
Duplicate orders  : 0
Late orders       : 36048
Late rate         : 57.3 %


## 15. Save the clean files
`lineterminator="\n"` writes simple line endings so MySQL reads the files the same way on any computer.

In [18]:
import os
os.makedirs("../data/processed", exist_ok=True)

orders.to_csv("../data/processed/orders_clean.csv", index=False, lineterminator="\n")
order_lines.to_csv("../data/processed/order_lines_clean.csv", index=False, lineterminator="\n")

print("Saved orders_clean.csv      :", len(orders), "rows")
print("Saved order_lines_clean.csv :", len(order_lines), "rows")

Saved orders_clean.csv      : 62897 rows
Saved order_lines_clean.csv : 172765 rows


## Summary

| Step | Result |
|---|---|
| Raw data | 180,519 order lines, 53 columns |
| Missing values | only in 4 columns we don't use |
| Duplicates | none |
| Columns kept | 18 (personal data removed) |
| "Late" | confirmed: actual days > promised days |
| Cancelled orders | removed (7,754 lines) |
| Shipping info within an order | always the same, so safe to combine |
| **orders_clean.csv** | **62,897 orders**, 16 columns |
| **order_lines_clean.csv** | **172,765 lines**, 9 columns |

**Next:** `sql/00_create_tables_and_load.sql` loads these two files into MySQL for the business analysis.